# Lab 01 — Model Anatomy: Tokens, Logits, and Generation

**Goal:** understand the object we are about to train.

Wordle is unusually useful because it exposes a mismatch: humans naturally reason over letters, while an LLM reasons over tokens. We want to *measure* that mismatch before trying to repair it.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tiny_wordle.hardware import preferred_device

MODEL_ID = "Qwen/Qwen3-0.6B"
device = preferred_device()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(device)
model.eval()

## 1.1 What is a word to the tokenizer?

Tokenize several Wordle words. Do not assume one English word equals one token.

In [ ]:
words = ["CRANE", "SLATE", "AUDIO", "QUEUE", "FUZZY", "jazzy", "crane"]

for word in words:
    ids = tokenizer.encode(word, add_special_tokens=False)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{word:8s} -> ids={ids} pieces={pieces}")

Now force an explicit character representation.

In [ ]:
representations = [
    "CRANE",
    "C R A N E",
    "[C][R][A][N][E]",
    "C|R|A|N|E",
]

for text in representations:
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"{text:20s} -> {len(ids):2d} tokens -> {pieces}")

### Observation

Record which representation most closely exposes individual letters to the model.

Important: more tokens is not automatically better. We are identifying a representational variable we may test later.

## 1.2 Token probabilities are the primitive

Ask a simple next-token question and inspect the probability distribution directly.

In [ ]:
prompt = "The five-letter English word C R A N"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**inputs).logits[0, -1]

probs = torch.softmax(logits.float(), dim=-1)
values, ids = torch.topk(probs, 15)

for p, token_id in zip(values.tolist(), ids.tolist()):
    print(f"{p:8.5f} {tokenizer.decode([token_id])!r}")

## 1.3 Greedy vs sampling

Generation is a policy over those token probabilities. Compare deterministic greedy decoding with sampling.

In [ ]:
def render_prompt(user_text: str, thinking: bool = False) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )

def generate(user_text: str, *, thinking=False, do_sample=False, temperature=0.7, top_p=0.8, seed=0):
    torch.manual_seed(seed)
    prompt = render_prompt(user_text, thinking=thinking)
    batch = tokenizer(prompt, return_tensors="pt").to(device)
    kwargs = dict(max_new_tokens=120, do_sample=do_sample)
    if do_sample:
        kwargs.update(temperature=temperature, top_p=top_p, top_k=20)
    with torch.no_grad():
        output = model.generate(**batch, **kwargs)
    new = output[0, batch["input_ids"].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True)

question = "Give exactly one legal five-letter Wordle guess. Output only the word."

print("GREEDY:")
print(generate(question, do_sample=False))

print("\nSAMPLED:")
for seed in range(3):
    print(seed, generate(question, do_sample=True, seed=seed))

## 1.4 Thinking mode is an experimental variable

Qwen3 supports a thinking mode. We are **not** deciding yet whether it helps Wordle.

Run the same prompt with thinking enabled and disabled. Keep the distinction in mind because later training/evaluation must not accidentally change this variable.

In [ ]:
prompt = "We are playing Wordle. Previous guess CRANE produced: C gray, R gray, A yellow, N green, E gray. Suggest the next guess and explain briefly."

print("NON-THINKING MODE")
print(generate(prompt, thinking=False, do_sample=True, seed=7))

print("\nTHINKING MODE")
print(generate(prompt, thinking=True, do_sample=True, temperature=0.6, top_p=0.95, seed=7))

## 1.5 Baseline failure probes

We are not building the full benchmark yet. We are looking for obvious failure modes.

Run these prompts and classify each response:

- valid five-letter word?
- respects green positions?
- respects yellow letters?
- avoids known absent letters?
- obeys requested output format?

In [ ]:
probes = [
    "Give exactly one legal five-letter Wordle guess. Output only the word.",
    "Wordle state: _ _ A _ _. Letter R is present but not position 2. C,E,S,T are absent. Give one next guess only.",
    "Wordle state: first letter is B, last letter is Y. A,E,I,O are absent. Give one next guess only.",
    "Wordle: guess EERIE gives feedback gray, yellow, gray, gray, green. Give one next guess only.",
]

for i, p in enumerate(probes, 1):
    print(f"\n--- Probe {i} ---")
    print(generate(p, thinking=False, do_sample=False))

## Lab 01 checkpoint

Write a short experiment note:

- What does Qwen's tokenizer do to five-letter words?
- Which failure mode seems most common?
- Does explicit character formatting appear worth testing later?
- What changes when thinking mode is enabled?
- What would you need to automate before calling any of this a benchmark?

The next lab finally changes weights.